# 시설 마스터

- 문화누리카드 가맹점 정리
- 일반 문화시설 정리
- 문화누리 중분류 기준으로 외부데이터 매핑


In [ ]:
from pathlib import Path
import re
import warnings

import numpy as np
import pandas as pd

try:
    import geopandas as gpd
except ImportError as e:
    raise ImportError("data314 커널에서 실행 필요: geopandas 없음") from e

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 100)

ROOT = Path.cwd()
while not (ROOT / "data").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent

RAW = ROOT / "data" / "raw"
PROCESSED = ROOT / "data" / "processed" / "table_design"
PROCESSED.mkdir(parents=True, exist_ok=True)

WGS84 = "EPSG:4326"
TARGET_CRS = "EPSG:5179"

WALK_CATS = ["도서", "문화체험", "음악", "영상", "체육시설", "체육용품"]
TRANSIT_CATS = ["미술", "공연", "스포츠관람", "관광지"]
DROP_CATS = ["숙박", "여행사", "교통수단"]


In [ ]:
# 저장 함수
def save_table(df, path):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    df.to_csv(path, index=False, encoding="utf-8-sig")
    return path


def save_gpkg(gdf, path, layer):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    gdf.to_file(path, layer=layer, driver="GPKG")
    return path


def clean_text(s):
    return s.fillna("").astype(str)


def point_gdf(df, lon_col="lon", lat_col="lat"):
    df = df.copy()
    df[lon_col] = pd.to_numeric(df[lon_col], errors="coerce")
    df[lat_col] = pd.to_numeric(df[lat_col], errors="coerce")
    df = df[df[lon_col].between(126.6, 127.4) & df[lat_col].between(37.2, 37.9)].copy()
    return gpd.GeoDataFrame(
        df,
        geometry=gpd.points_from_xy(df[lon_col], df[lat_col]),
        crs=WGS84,
    ).to_crs(TARGET_CRS)


def add_access_mode(df):
    df = df.copy()
    df["access_mode"] = np.where(
        df["mnc_middle_cat"].isin(WALK_CATS),
        "walk",
        np.where(df["mnc_middle_cat"].isin(TRANSIT_CATS), "transit", "drop"),
    )
    return df


## 매핑 기준

- 도보: 도서, 문화체험, 음악, 영상, 체육시설, 체육용품
- 대중교통: 미술, 공연, 스포츠관람, 관광지
- 제거: 숙박, 여행사, 교통수단


In [ ]:
# 문화누리 중분류 - 외부데이터 매핑
mapping_rows = [
    ["도서", "문화예술공간 API", "도서관"],
    ["도서", "상권데이터", "서점, 만화방"],
    ["문화체험", "문화예술공간 API", "문화/복지/시군구회관"],
    ["문화체험", "상권데이터", "문화센터, 공방, 공예, 체험 등 키워드"],
    ["음악", "상권데이터", "악기 소매업, 음반/비디오물 소매·대여, 악기 수리"],
    ["영상", "문화누리 가맹점", "영화만 유지, TV 제거"],
    ["체육시설", "전국체육시설 API", "정상운영 + 좌표 있음"],
    ["체육용품", "상권데이터", "운동용품, 자전거, 스포츠/레크리에이션 용품"],
    ["미술", "문화예술공간 API", "미술관, 박물관"],
    ["미술", "상권데이터", "사진촬영, 사진기/광학기기, 화방, 공방, 공예 키워드"],
    ["공연", "문화예술공간 API", "공연장"],
    ["스포츠관람", "스포츠관람 POI", "경기장 후보 중복제거 파일"],
    ["관광지", "관광공사 TourAPI", "contentTypeId=12 관광지"],
    ["숙박", "제거", "접근성 분석 제외"],
    ["여행사", "제거", "접근성 분석 제외"],
    ["교통수단", "제거", "접근성 분석 제외"],
]

mapping_table = pd.DataFrame(mapping_rows, columns=["mnc_middle_cat", "source", "rule"])
save_table(mapping_table, PROCESSED / "facility_mapping_table.csv")
mapping_table


## 문화누리카드 가맹점

- 오프라인 가맹점 사용
- 숙박, 여행사, 교통수단 제거
- 영상은 영화만 유지


In [ ]:
# 문화누리카드 가맹점 전처리
mnc_path = RAW / "merchants" / "source" / "mnc_seoul_offline_merchants_20260706.xlsx"
mnc = pd.read_excel(mnc_path)
mnc = mnc[mnc["가맹점명"].notna()].copy()
mnc = mnc.rename(
    columns={
        "가맹점명": "facility_name",
        "분야": "mnc_large_cat",
        "Unnamed: 4": "mnc_middle_cat",
        "Unnamed: 5": "mnc_sub_cat",
        "위도": "lat",
        "경도": "lon",
        "지역": "sido_nm",
        "Unnamed: 12": "gu_nm",
        "주소": "address",
    }
)

mnc = mnc[~mnc["mnc_middle_cat"].isin(DROP_CATS)].copy()
mnc = mnc[~((mnc["mnc_middle_cat"].eq("영상")) & (mnc["mnc_sub_cat"].eq("TV")))].copy()

mnc["facility_set"] = "mnc"
mnc["source"] = "문화누리카드 가맹점"
mnc["facility_id"] = "mnc_" + mnc.index.astype(str)
mnc = add_access_mode(mnc)

mnc_gdf = point_gdf(mnc, "lon", "lat")
mnc_gdf = mnc_gdf[mnc_gdf["access_mode"].ne("drop")].copy()

mnc_keep = [
    "facility_id", "facility_set", "source", "facility_name", "mnc_large_cat",
    "mnc_middle_cat", "mnc_sub_cat", "access_mode", "sido_nm", "gu_nm",
    "address", "lon", "lat", "geometry",
]
mnc_gdf = mnc_gdf[mnc_keep]

mnc_gdf["mnc_middle_cat"].value_counts()


## 일반 문화시설

- 공식 시설 데이터 우선
- 부족한 분류만 상권데이터 보완


In [ ]:
# 문화예술공간 API
culture = pd.read_csv(RAW / "facilities" / "culture_facilities" / "parsed" / "culture_facilities_combined.csv")
culture_map = {
    "도서관": "도서",
    "문화/복지/시군구회관": "문화체험",
    "미술관": "미술",
    "박물관": "미술",
    "공연장": "공연",
}
culture = culture[culture["culGrpName"].isin(culture_map)].copy()
culture["mnc_middle_cat"] = culture["culGrpName"].map(culture_map)
culture["facility_set"] = "general"
culture["source"] = "문화예술공간 API"
culture["facility_name"] = culture["culName"]
culture["facility_id"] = "culture_" + culture["source_endpoint"].astype(str) + "_" + culture["seq"].astype(str)
culture = add_access_mode(culture)
culture_gdf = point_gdf(culture, "lon", "lat")

# 전국체육시설 API
sports = pd.read_csv(
    RAW / "facilities" / "sports_facilities" / "parsed" / "sports_facilities_seoul_addr_or_bbox_candidates.csv",
    low_memory=False,
)
sports = sports[sports["faci_stat_nm"].eq("정상운영")].copy()
sports["mnc_middle_cat"] = "체육시설"
sports["facility_set"] = "general"
sports["source"] = "전국체육시설 API"
sports["facility_name"] = sports["faci_nm"]
sports["mnc_sub_cat"] = sports["fcob_nm"].fillna("") + " / " + sports["ftype_nm"].fillna("")
sports["facility_id"] = "sports_" + sports["faci_cd"].astype(str)
sports = add_access_mode(sports)
sports_gdf = point_gdf(sports, "lon", "lat")

# 스포츠관람 POI
spectator = pd.read_csv(
    RAW / "facilities" / "sports_spectator" / "parsed" / "sports_spectator_seoul_access_candidates_dedup.csv",
    low_memory=False,
)
spectator["mnc_middle_cat"] = "스포츠관람"
spectator["facility_set"] = "general"
spectator["source"] = "스포츠관람 POI"
spectator["facility_name"] = spectator["poi_nm"]
spectator["mnc_sub_cat"] = spectator["mcate_nm"]
spectator["facility_id"] = "spectator_" + spectator["id"].astype(str)
spectator = add_access_mode(spectator)
spectator_gdf = point_gdf(spectator, "lon", "lat")

# 관광공사 TourAPI
tourism = pd.read_csv(
    RAW / "facilities" / "tourism_attractions" / "parsed" / "tourism_attractions_seoul_contentTypeId_12_with_category_names.csv",
    low_memory=False,
)
tourism["mnc_middle_cat"] = "관광지"
tourism["facility_set"] = "general"
tourism["source"] = "관광공사 TourAPI"
tourism["facility_name"] = tourism["title"]
tourism["mnc_sub_cat"] = tourism["cat2_name"].fillna("") + " / " + tourism["cat3_name"].fillna("")
tourism["facility_id"] = "tourism_" + tourism["contentid"].astype(str)
tourism = add_access_mode(tourism)
tourism_gdf = point_gdf(tourism, "lon", "lat")


In [ ]:
# 상권데이터 보완
shop_path = RAW / "franchise_candidates" / "source" / "small_business_market_area_seoul_202603.csv"
shop = pd.read_csv(shop_path, low_memory=False)

for col in ["상호명", "상권업종대분류명", "상권업종중분류명", "상권업종소분류명", "표준산업분류명"]:
    shop[col] = clean_text(shop[col])

shop["text"] = (
    shop["상호명"] + " " + shop["상권업종대분류명"] + " " + shop["상권업종중분류명"] + " "
    + shop["상권업종소분류명"] + " " + shop["표준산업분류명"]
)

# 도서
book_mask = shop["상권업종소분류명"].str.contains("서점|만화방", regex=True)
book_mask &= ~shop["text"].str.contains("독서실|스터디|고시원", regex=True)

# 문화체험
exp_mask = shop["text"].str.contains("문화센터|공방|공예|도예|체험|레크리에이션", regex=True)
exp_mask &= ~shop["text"].str.contains("노래방|PC방|전자 게임장|복권|독서실|스터디", regex=True)

# 음악
music_mask = shop["text"].str.contains("악기 소매|음반/비디오물|악기 수리", regex=True)
music_mask &= ~shop["text"].str.contains("노래방|음악학원", regex=True)

# 체육용품
goods_mask = shop["text"].str.contains("운동용품|자전거 소매|스포츠/레크리에이션 용품", regex=True)
goods_mask &= ~shop["text"].str.contains("낚시터 운영", regex=True)

# 미술
art_mask = shop["text"].str.contains("사진촬영|사진기/기타 광학기기|화방|공방|공예|갤러리", regex=True)
art_mask &= ~shop["text"].str.contains("미술학원|입시|교과학원", regex=True)

shop_rules = [
    ("도서", book_mask),
    ("문화체험", exp_mask),
    ("음악", music_mask),
    ("체육용품", goods_mask),
    ("미술", art_mask),
]

shop_parts = []
for cat, mask in shop_rules:
    tmp = shop[mask].copy()
    tmp["mnc_middle_cat"] = cat
    tmp["facility_set"] = "general"
    tmp["source"] = "상권데이터"
    tmp["facility_name"] = tmp["상호명"]
    tmp["mnc_sub_cat"] = tmp["상권업종소분류명"]
    tmp["facility_id"] = "shop_" + cat + "_" + tmp["상가업소번호"].astype(str)
    tmp = add_access_mode(tmp)
    tmp = tmp.rename(columns={"시군구명": "gu_nm", "도로명주소": "address", "경도": "lon", "위도": "lat"})
    shop_parts.append(tmp)

shop_fac = pd.concat(shop_parts, ignore_index=True)
shop_gdf = point_gdf(shop_fac, "lon", "lat")

shop_gdf["mnc_middle_cat"].value_counts()


In [ ]:
# 일반 시설 통합
common_cols = [
    "facility_id", "facility_set", "source", "facility_name", "mnc_middle_cat",
    "mnc_sub_cat", "access_mode", "gu_nm", "address", "lon", "lat", "geometry",
]

for gdf in [culture_gdf, sports_gdf, spectator_gdf, tourism_gdf, shop_gdf]:
    for col in common_cols:
        if col not in gdf.columns:
            gdf[col] = np.nan

general_gdf = pd.concat(
    [culture_gdf[common_cols], sports_gdf[common_cols], spectator_gdf[common_cols], tourism_gdf[common_cols], shop_gdf[common_cols]],
    ignore_index=True,
)
general_gdf = gpd.GeoDataFrame(general_gdf, geometry="geometry", crs=TARGET_CRS)

# 좌표 기준 중복 제거
general_gdf["lon_round"] = general_gdf["lon"].round(6)
general_gdf["lat_round"] = general_gdf["lat"].round(6)
general_gdf = general_gdf.drop_duplicates(
    subset=["facility_set", "source", "mnc_middle_cat", "facility_name", "lon_round", "lat_round"]
).drop(columns=["lon_round", "lat_round"])

# 영상 일반시설은 보류
mapping_gap = pd.DataFrame(
    [
        {
            "mnc_middle_cat": "영상",
            "gap": "일반 영화관 좌표 원자료 필요",
            "note": "문화누리 가맹점은 영화만 유지. 일반시설 비교는 영화관 데이터 확보 후 추가.",
        }
    ]
)

save_table(mapping_gap, PROCESSED / "facility_mapping_gap.csv")
save_table(general_gdf.drop(columns="geometry"), PROCESSED / "facility_general_master.csv")
save_table(mnc_gdf.drop(columns="geometry"), PROCESSED / "facility_mnc_master.csv")
save_gpkg(general_gdf, PROCESSED / "facility_general_master.gpkg", "general")
save_gpkg(mnc_gdf, PROCESSED / "facility_mnc_master.gpkg", "mnc")

summary = pd.concat(
    [
        general_gdf.groupby(["facility_set", "mnc_middle_cat", "source"], as_index=False).size(),
        mnc_gdf.groupby(["facility_set", "mnc_middle_cat", "source"], as_index=False).size(),
    ],
    ignore_index=True,
)
save_table(summary, PROCESSED / "facility_master_summary.csv")
summary.sort_values(["facility_set", "mnc_middle_cat", "source"])
